In [1]:
import sys

project_root = 'c:/big20/git/big20-ML-project2-team3/SantanderCS'

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    
import numpy as np
import pandas as pd
from sklearn.ensemble    import RandomForestClassifier
from sklearn.metrics     import classification_report, roc_auc_score, f1_score, recall_score
from utils.preprocessing import load_data, split_features_target, scale_data, data_split, remove_zero_columns2
from utils.user_utils    import get_model_train_eval
from utils.feature_engineering import add_statistical_features, drop_highly_correlated_features


In [2]:
# 데이터 로딩 및 기본 전처리
train, test = load_data()


In [3]:
def undersample(X, y, ratio, random_state=23):
    """Random undersampling of majority class to achieve ratio (pos:neg = 1:ratio)
       X: DataFrame, y: Series with index aligned to X
    """
    pos_idx = y[y == 1].index
    neg_idx = y[y == 0].index

    n_pos = len(pos_idx)
    n_neg_sample = int(n_pos * ratio)
    if n_neg_sample > len(neg_idx):
        raise ValueError(f"Requested neg samples {n_neg_sample} > available {len(neg_idx)}")

    neg_sampled_idx = pd.Series(neg_idx).sample(n=n_neg_sample, random_state=random_state).values

    selected_idx = np.concatenate([pos_idx.values, neg_sampled_idx])
    np.random.RandomState(seed=random_state).shuffle(selected_idx)

    X_bal = X.loc[selected_idx].reset_index(drop=True)
    y_bal = y.loc[selected_idx].reset_index(drop=True)

    return X_bal, y_bal

In [4]:
# ID와 Target 분리하기 
X_features, y_labels = split_features_target(train)
X_test = test.drop(columns=['ID'], axis=1)

In [5]:
# zero_count_rate 제거했을때 제거될 컬럼수 149개 잔존
X_features, X_test = remove_zero_columns2(X_features, X_test, 0.99, False)



Train Data Analysis (Threshold: 99.0% )
Train rows: 76,020, columns: 369
Test rows: 75,818, columns: 369

                                   Train Summary (zero_count 내림차순)                                    
                   ColumnName  na_Sum  nUnique          mode  modeFreq modeFreqRate  zero_count zero_count_rate_display
saldo_medio_var13_medio_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
                     ind_var2       0        1      0.000000     76020      100.00%       76020                 100.00%
        num_reemb_var33_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
    num_trasp_var17_out_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
    num_trasp_var33_out_hace3       0        1      0.000000     76020      100.00%       76020                 100.00%
              saldo_var2_ult1       0        1      0.000000     76020

In [ ]:
print(X_features.shape)

# underSampling
ratio = 20 #[5, 10, 20]
X_Balance, y_balance = undersample(X_features, y_labels, ratio)
print(X_Balance.shape, y_balance.shape)
rt = '''
5:
(76020, 149)
(18048, 149) (18048,)

10: 
(76020, 149)
(33088, 149) (33088,)

20:
(63168, 149) (63168,)
'''

(76020, 149)
(63168, 149) (63168,)


In [50]:
# var3 처리
X_features['var3'] = X_features['var3'].replace(-999999, 2)

In [ ]:
# 상관계수 높은 feature들 삭제하기
# X_reduced, to_drop = drop_highly_correlated_features(X_features)
X_reduced, to_drop = drop_highly_correlated_features(X_Balance)
X_test_reduced = X_test.drop(to_drop, axis=1)


In [52]:
print(X_reduced.shape, X_test_reduced.shape)

(18048, 94) (75818, 94)


In [ ]:
# 삭제된 컬럼 개수 확인
print("Train에서 삭제된 컬럼 개수:", len(to_drop))
# radio=삭제된 컬럼수 : 5=55, 10=57, 20=54 

Train에서 삭제된 컬럼 개수: 55


In [ ]:
# 삭제된 컬럼 목록 조회
# for i, col in enumerate(sorted(to_drop), start=1):
#     print(f"{i:>2} : {col}")

# 리스트를 Pandas Series로 변환
series = pd.Series(sorted(to_drop), name="Dropped_Columns")

# CSV 파일로 저장
series.to_csv(f"../data/99perCorr95UnderSampling{ratio}DroppedColumns_20251123.xls", index=False)


In [ ]:
# 스케일링 
# X_train_scaled, X_test_scaled, scaler = scale_data(X_reduced, X_test_reduced)

In [55]:
# 학습/테스트 데이터 분리
# X_train, X_val, y_train, y_val = data_split(X_train_scaled, y_labels)
X_train, X_val, y_train, y_val = data_split(X_reduced, y_balance)

In [ ]:
# model_name = 'RandomForest_99per_corrTh95_NoTScaled_ne390_maxDepth25_classWeight12_msl1_mss7' 
# model_name = 'RandomForest_99per_corrTh95_Scaled_ne390_maxDepth25_classWeight12_msl1_mss7' 
model_name = f'RandomForest_99per_corrTh95UnderSampling{ratio}_BestOpt' 
# Best Option 적용
rf_clf = RandomForestClassifier(
  random_state = 0,
  n_estimators = 390,
  max_depth    = 25, 
  class_weight = {0:1, 1:2}, # 클래스별 가중치
  min_samples_leaf  = 1, 
  min_samples_split = 7,
  n_jobs            = -1 # 병렬처리 여부     
)

# 함수 이용
get_model_train_eval(rf_clf, model_name, X_train, X_val, y_train, y_val)
result_text = '''
✓ 모델 저장 완료: ../models\RandomForest_99per_corrTh95UnderSampling5_BestOpt.pkl
  파일 크기: 56.93 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8306, 정확도: 0.8407, 정밀도: 0.5256, 재현율: 0.4601, F1: 0.4907
오차행렬:
[[2758  250]
 [ 325  277]]
실행 시간: 1.8377294540405273

✓ 모델 저장 완료: ../models\RandomForest_99per_corrTh95UnderSampling10_BestOpt.pkl
  파일 크기: 66.40 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8255, 정확도: 0.9004, 정밀도: 0.4300, 재현율: 0.2907, F1: 0.3469
오차행렬:
[[5784  232]
 [ 427  175]]
실행 시간: 3.1548142433166504

✓ 모델 저장 완료: ../models\RandomForest_99per_corrTh95UnderSampling20_BestOpt.pkl
  파일 크기: 77.76 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8230, 정확도: 0.9515, 정밀도: 0.3659, 재현율: 0.0249, F1: 0.0467
오차행렬:
[[12006    26]
 [  587    15]]
실행 시간: 7.465203285217285

'''

✓ 모델 저장 완료: ../models\RandomForest_99per_corrTh95UnderSampling5_BestOpt.pkl
  파일 크기: 56.93 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8306, 정확도: 0.8407, 정밀도: 0.5256, 재현율: 0.4601, F1: 0.4907
오차행렬:
[[2758  250]
 [ 325  277]]
실행 시간: 1.8377294540405273


In [ ]:
#threshold  dropped_features  AUC       F1        Recall
# 0.95      55                0.8306    0.4907    0.4601   (UnderSampling 50% Not Scale)
# 0.95      57                0.8255    0.3469    0.2907   (UnderSampling 10% Not Scale)
# 0.95      54                0.8230    0.0467    0.0249   (UnderSampling 20% Not Scale)
# 0.95      54                0.8230    0.0467    0.0249   (UnderSampling 20% Scale)
# 0.95      53                0.842169  0.016287  0.008306 (data Not Scaled) ***
# 0.95      53                0.8409    0.0131    0.0066   (Scaled)
# 0.95      57                0.8359    0.0350    0.0183   (Not Scaled)


In [ ]:
# # 비교 시각화 
# import pickle
# import matplotlib.pyplot as plt
# import seaborn as sns
# from sklearn.metrics import roc_curve, auc, precision_recall_curve, confusion_matrix


# # 2. 예측 확률 구하기 (y_true는 실제 라벨)
# y_pred_proba_53 = model_53.predict_proba(X_test)[:, 1]
# y_pred_proba_57 = model_57.predict_proba(X_test)[:, 1]

# # 3. ROC, PR, Confusion Matrix 시각화 함수
# def plot_all(y_true, y_pred_proba, threshold=0.95, title="Model"):
#     fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
#     roc_auc = auc(fpr, tpr)

#     precision, recall, _ = precision_recall_curve(y_true, y_pred_proba)

#     y_pred = (y_pred_proba >= threshold).astype(int)
#     cm = confusion_matrix(y_true, y_pred)

#     fig, axes = plt.subplots(1, 3, figsize=(18, 5))

#     # ROC Curve
#     axes[0].plot(fpr, tpr, color="blue", lw=2, label=f"AUC = {roc_auc:.3f}")
#     axes[0].plot([0, 1], [0, 1], color="gray", linestyle="--")
#     axes[0].set_title(f"ROC Curve - {title}")
#     axes[0].set_xlabel("False Positive Rate")
#     axes[0].set_ylabel("True Positive Rate")
#     axes[0].legend(loc="lower right")

#     # Precision-Recall Curve
#     axes[1].plot(recall, precision, color="green", lw=2)
#     axes[1].set_title(f"Precision-Recall Curve - {title}")
#     axes[1].set_xlabel("Recall")
#     axes[1].set_ylabel("Precision")

#     # Confusion Matrix
#     sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[2])
#     axes[2].set_title(f"Confusion Matrix - {title}")
#     axes[2].set_xlabel("Predicted")
#     axes[2].set_ylabel("Actual")

#     plt.tight_layout()
#     plt.show()

# # 4. 두 모델 비교 실행
# plot_all(y_true, y_pred_proba_53, threshold=0.95, title="Dropped 53 Features")
# plot_all(y_true, y_pred_proba_57, threshold=0.95, title="Dropped 57 Features")